In [50]:
import pandas as pd 
import os
from tensorflow import keras
from keras.layers import Dense,Flatten
from keras import Sequential
from keras.applications.vgg16  import VGG16

In [51]:
import tensorflow as tf
train_folder = r"C:\Users\Krish\Downloads\Deep Learning\TransferLearning\train"
test_folder = r"C:\Users\Krish\Downloads\Deep Learning\TransferLearning\test"

## Loading train and validation images
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_folder,
    labels= 'inferred',
    image_size = (224,224),
    batch_size = 32,
    label_mode = 'int',
    shuffle = True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    test_folder, 
    image_size = (224,224),
    batch_size = 32,
    label_mode = 'int',
    labels = 'inferred'
)


Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


In [52]:
## Normalize images 
def process(img, label):
    image = tf.cast(img/255., tf.float32 )
    return image,label

train_ds = train_ds.map(process)
val_ds = val_ds.map(process)

In [53]:
type(train_ds)

tensorflow.python.data.ops.map_op._MapDataset

In [62]:
## 
conv_base = VGG16(
    weights="imagenet",
    include_top= False,
    input_shape=(224,224,3)
)

In [63]:
conv_base.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

In [64]:
model = Sequential()
model.add(conv_base)
model.add(Flatten())
model.add(Dense(256,activation= "relu"))
model.add(Dense(1, activation = "sigmoid"))


In [65]:
model.compile(optimizer= "adam", loss= "binary_crossentropy", metrics= ["accuracy"])

In [66]:
conv_base.trainable = False

In [67]:
model.fit(train_ds, validation_data= val_ds, epochs = 10)

Epoch 1/10
 22/625 ━━━━━━━━━━━━━━━━━━━━ 31:14 3s/step - accuracy: 0.5885 - loss: 2.4305

KeyboardInterrupt: 